In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In this notebook, I'll analyze the behavior from epilepsy patients. 

12/18: I run into a consistent issue with mismatched learn_df and combined_df trials..... 

1/15/24: I have screwed up the Day 2 memory task. Instead of ensuring that the images from Day 1 do not appear on Day 2, about half of the old ones do. 

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import glob as glob
import arviz as az
import pandas as pd
import pickle 
import sys 
sys.path.append('/hpc/users/qasims01/resources/ImgRatingSorter/')
from util import dprime
sys.path.append('/hpc/users/qasims01/resources/utils/RLModels_K99/scripts')
import param_recov_utils
import model_recov_utils
from model_utils import *
from RWModel import RWModel
from WSLSModel import WSLSModel
from BayesModel import BayesModel
from RWModel_Fictive1 import RWModel_Fictive1
from RWModel_Fictive2 import RWModel_Fictive2
from RWModel_Signed import RWModel_Signed
from EWAModel import EWA
from VKFModel import VKFModel
import pymc as pm

# Hierarchical models
from HierarchicalVKF import HierarchicalVKFModel
from HierarchicalRW import HierarchicalRWModel
from HierarchicalBayes import HierarchicalBayesModel
from HierarchicalRW_Signed import HierarchicalRWModel_Signed
from HierarchicalRW_Fictive1 import HierarchicalRWModel_Fictive1

from matplotlib.backends.backend_pdf import PdfPages

# plt.style.use('seaborn-paper')
plt.rc("axes.spines", top=False, right=False)
plt.rc("axes.spines", top=False, right=False)
plt.rc('lines', lw=1, c='r')
# plt.rc('font',**{'family':'sans-serif','sans-serif':['Arial']})

custom_palette = ["#b65975", "#59b675"]
# sns.set_palette(custom_palette)

In [ ]:
import time, sys
from IPython.display import clear_output

def update_progress(progress):
    bar_length = 20
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
    if progress < 0:
        progress = 0
    if progress >= 1:
        progress = 1

    block = int(round(bar_length * progress))

    clear_output(wait = True)
    text = "Progress: [{0}] {1:.1f}%".format( "#" * block + "-" * (bar_length - block), progress * 100)
    print(text)

In [ ]:
# test with a patient (MS008):

In [ ]:
base_dir = '/sc/arion'
load_dir = f'{base_dir}/projects/guLab/Salman/EMU'
save_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

In [ ]:
# Load the demographics data to get information about who has run the task
demo_file = f'{load_dir}/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file)


In [ ]:

subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']
# subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
full_data_df = []
learn_df = []

# iterate through the subjects 
for ix, cell, in subj_df.iterrows(): 
    
    subj = cell.MSSMCode.strip(' ')
    
    if cell.LongVersion==1:
        two_ab_ntrials = 80
        all_mem_file =f'{load_dir}/all_mem_data_long.xlsx' 
        all_mem_df = pd.read_excel(all_mem_file, engine='openpyxl')
        all_mem_df = all_mem_df[['img_path', 'DPRIME']]
        all_mem_df.img_path = all_mem_df.img_path.apply(lambda x: x[:-4]+'.png' )
    
    elif cell.LongVersion==0:
        
        two_ab_ntrials = 60
        all_mem_file =f'{load_dir}/all_mem_data_short.xlsx' 
        all_mem_df = pd.read_excel(all_mem_file, engine='openpyxl')
        all_mem_df = all_mem_df[['img_path', 'DPRIME']]
        all_mem_df.img_path = all_mem_df.img_path.apply(lambda x: x[:-4]+'.png' )
        
    
    
    day1_df = pd.read_csv(glob.glob(f'{load_dir}/{subj}/behav/Day1/*.csv')[0], index_col=None, header=0)
    
    # trim down to relevant columns 
    relevant_cols = ['MB1_draw_key.keys', 'MB1_draw_key.started', 'MB1_draw_key.rt','img_path', 'prob', 'block', 'Gender',
           'reward', 'participant', 'MB1_face.started', 'MB1_ITI_cross.started']
    
    # For some reason the trials and the data are coded in alternating rows
    trials = (day1_df.iloc[2:(two_ab_ntrials*2)+2:2]['trials_2.thisN'].astype(int)+1).tolist()
    mb_df = day1_df.iloc[2:(two_ab_ntrials*2)+2:2][relevant_cols].reset_index(drop=True)
    
    mb_df['trials'] = trials
    mb_df.rename(columns={'MB1_draw_key.keys': 'gamble',
                 'MB1_draw_key.started': 'gamble_start',
                 'MB1_face.started': 'feedback_start',
                 'MB1_ITI_cross.started': 'baseline_start',
                 'MB1_draw_key.rt': 'gamble_rt',
                 'Gender': 'gamble_label'},
                 inplace=True)
    mb_df['baseline_start'] = day1_df.iloc[3:(two_ab_ntrials*2)+3:2]['MB1_ITI_cross.started'].values.tolist()
    
    # Code reward as 1 or 0
    mb_df.reward = mb_df.reward/100
    # Code gamble as 1 or 0 (0 is male, 1 is female)
    mb_df.gamble = mb_df.gamble.apply(lambda x: 0 if x==1.0 else 1)
    mb_df.gamble_label = mb_df.gamble_label.apply(lambda x: 'male' if x==1.0 else 'female')

#     mb_df['age'] = demographics_df[demographics_df.participant==mb_df.participant[0]].age.unique().tolist()*mb_df['trials'].max()
#     mb_df['sex'] = demographics_df[demographics_df.participant==mb_df.participant[0]].Sex.unique().tolist()*mb_df['trials'].max()


    mb_df = mb_df.merge(all_mem_df, on='img_path', how='left', indicator=False)
#     mb_df.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 
    
    # Add a filter for people with bad performance
    mb_df['good_dm_performance'] = True
    rew = mb_df.reward.sum()
    if rew<=(two_ab_ntrials//2): # person was worse than chance
        mb_df.good_dm_performance=False
        
    # Now, process the memory data
    
    if cell.LongVersion==1:

        relevant_cols = ['img_path', 'prob', 'block', 'Gender',
               'reward', 'participant', 'MEM2_recall_key.keys', 'MEM2_recall_key.rt', 'MEM2_recall_key.stopped', 'MEM2_stimuli.started',
               'MEM2_conf_slider.rating', 'MEM2_conf_slider.rt','trials_3.thisN', 'MEM2_jitter.started', 'trialIm']
        rm_df_day1 = day1_df.dropna(subset=['trials_3.thisN'])[relevant_cols].reset_index(drop=True)
        
        rm_df_day1['img_path'] = day1_df.img_path.dropna()[-80:]
        rm_df_day1['prob'] = day1_df.prob.dropna()[-80:]
        rm_df_day1['block'] = day1_df.block.dropna()[-80:]
        rm_df_day1['Gender'] = day1_df.Gender.dropna()[-80:]
        rm_df_day1['reward'] = day1_df.reward.dropna()[-80:]
        rm_df_day1['participant'] = day1_df.participant.dropna()[-80:]
        rm_df_day1.rename(columns={'MEM2_recall_key.keys': 'recog_choice',
                     'MEM2_recall_key.rt': 'recog_rt',
                     'MEM2_conf_slider.rating': 'conf_choice',
                     'MEM2_conf_slider.rt': 'conf_rt',
                     'MEM2_recall_key.stopped': 'recog_time', 
                     'MEM2_jitter.started': 'baseline_start',
                     'MEM2_stimuli.started': 'cue_on',
                     'trials_3.thisN': 'trials',
                     'Gender': 'gamble_label'},
                     inplace=True) 
        
        rm_df_day1.gamble_label = rm_df_day1.gamble_label.apply(lambda x: 'male' if x==1.0 else 'female')
        # Code reward as 1 or 0
        rm_df_day1.reward = rm_df_day1.reward/100
        # Code recog_choice 
        rm_df_day1['recog_choice_bool'] = rm_df_day1.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
        rm_df_day1.recog_choice = rm_df_day1.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

        rm_df_day1.trials = rm_df_day1.trials.astype(int) + 1
        rm_df_day1['img_path'] = rm_df_day1['trialIm']
    #     rm_df['age'] = demographics_df[demographics_df.participant==mb_df.participant[0]].age.unique().tolist()*rm_df['trials'].max()
    #     rm_df['sex'] = demographics_df[demographics_df.participant==mb_df.participant[0]].Sex.unique().tolist()*rm_df['trials'].max()

        rm_df_merge_day1 = rm_df_day1.merge(mb_df, on='img_path', how='left', indicator=True, suffixes=(None, '_y'))

        # compute the hit-rates and false-alarm rates 
        hits = rm_df_merge_day1[(rm_df_merge_day1._merge=='both') & (rm_df_merge_day1.recog_choice_bool==1)].shape[0]
        misses = rm_df_merge_day1[(rm_df_merge_day1._merge=='both') & (rm_df_merge_day1.recog_choice_bool==0)].shape[0]

        false_alarms = rm_df_merge_day1[(rm_df_merge_day1._merge=='left_only') & (rm_df_merge_day1.recog_choice_bool==1)].shape[0]
        correct_rejections = rm_df_merge_day1[(rm_df_merge_day1._merge=='left_only') & (rm_df_merge_day1.recog_choice_bool==0)].shape[0]

        hm = hits+misses
        fc = false_alarms+correct_rejections

        hrs = hits/hm
        fars = false_alarms/fc

        dp = dprime(hrs, fars, hm, fc)

        rm_df_merge_day1['hits'] = (rm_df_merge_day1._merge=='both') & (rm_df_merge_day1.recog_choice_bool==1)
        rm_df_merge_day1['misses'] = (rm_df_merge_day1._merge=='both') & (rm_df_merge_day1.recog_choice_bool==0)
        rm_df_merge_day1['false_alarms'] = (rm_df_merge_day1._merge=='left_only') & (rm_df_merge_day1.recog_choice_bool==1)
        rm_df_merge_day1['correct_rejections'] = (rm_df_merge_day1._merge=='left_only') & (rm_df_merge_day1.recog_choice_bool==0)

        rm_df_merge_day1['hit_rate'] = hrs
        rm_df_merge_day1['false_alarm_rate'] = fars
        rm_df_merge_day1['dprime'] = np.nan

        if dp != float("-inf"):
            rm_df_merge_day1['dprime'] = dp 
            
        rm_df_merge_day1['condition'] = 'day_1'

        # Merge in the image DPRIME 
        rm_df_merge_day1 = rm_df_merge_day1.merge(all_mem_df, on='img_path', how='left', indicator=False)
        rm_df_merge_day1.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 
        
        
        # Process Day 2 
        if cell.Day2 == 1:
            rm_df_day2 = pd.read_csv(glob.glob(f'{load_dir}/{subj}/behav/Day2/*.csv')[0], index_col=None, header=0)

            rm_df_day2.rename(columns={'trialIm': 'img_path'}, inplace=True) 
                        
            num_dual_presentations = len(set(list(set(mb_df.img_path.values).intersection(set(rm_df_day1.img_path.dropna().values)))).intersection(set(rm_df_day2.img_path.dropna().values)))
            print(f'I made a mistake. {num_dual_presentations} images are presented for recall TWICE!')
            rm_df_day2['repeated_presentation'] = 0
            rm_df_day2['repeated_presentation'][rm_df_day2['img_path'].isin(rm_df_day1['img_path'])] =  1

            relevant_cols = ['img_path', 'participant', 'key_resp.keys', 'key_resp.rt', 'key_resp.stopped', 'image.started',
                       'new_slider.rating', 'new_slider.rt','trials.thisN', 'polygon.started', 'repeated_presentation']
            rm_df_day2 = rm_df_day2.dropna(subset=['trials.thisN'])[relevant_cols].reset_index(drop=True)

            rm_df_day2.rename(columns={'key_resp.keys': 'recog_choice',
                         'key_resp.rt': 'recog_rt',
                         'polygon.started': 'baseline_start',
                         'new_slider.rating': 'conf_choice',
                         'key_resp.stopped': 'recog_time',
                          'image.started': 'cue_on',
                         'new_slider.rt': 'conf_rt',
                         'trials.thisN': 'trials'},
                         inplace=True) 

    #         # baseline relative to first sync pulse: 
    #         rm_df_day2['baseline_start']  = rm_df_day2['baseline_start'] - sync_ts2
    #         rm_df_day2['recog_time']  = rm_df_day2['recog_time'] - sync_ts2

            # Code recog_choice 
            rm_df_day2['recog_choice_bool'] = rm_df_day2.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
            rm_df_day2.recog_choice = rm_df_day2.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

            rm_df_day2.trials = rm_df_day2.trials.astype(int) + 1
        #     rm_df['age'] = demographics_df[demographics_df.participant==mb_df.participant[0]].age.unique().tolist()*rm_df['trials'].max()
        #     rm_df['sex'] = demographics_df[demographics_df.participant==mb_df.participant[0]].Sex.unique().tolist()*rm_df['trials'].max()

            rm_df_merge_day2 = rm_df_day2.merge(mb_df, on='img_path', how='left', indicator=True, suffixes=(None, '_y'))

            # compute the hit-rates and false-alarm rates 
            hits = rm_df_merge_day2[(rm_df_merge_day2._merge=='both') & (rm_df_merge_day2.recog_choice_bool==1)].shape[0]
            misses = rm_df_merge_day2[(rm_df_merge_day2._merge=='both') & (rm_df_merge_day2.recog_choice_bool==0)].shape[0]

            false_alarms = rm_df_merge_day2[(rm_df_merge_day2._merge=='left_only') & (rm_df_merge_day2.recog_choice_bool==1)].shape[0]
            correct_rejections = rm_df_merge_day2[(rm_df_merge_day2._merge=='left_only') & (rm_df_merge_day2.recog_choice_bool==0)].shape[0]

            hm = hits+misses
            fc = false_alarms+correct_rejections

            hrs = hits/hm
            fars = false_alarms/fc

            dp = dprime(hrs, fars, hm, fc)

            rm_df_merge_day2['hits'] = (rm_df_merge_day2._merge=='both') & (rm_df_merge_day2.recog_choice_bool==1)
            rm_df_merge_day2['misses'] = (rm_df_merge_day2._merge=='both') & (rm_df_merge_day2.recog_choice_bool==0)
            rm_df_merge_day2['false_alarms'] = (rm_df_merge_day2._merge=='left_only') & (rm_df_merge_day2.recog_choice_bool==1)
            rm_df_merge_day2['correct_rejections'] = (rm_df_merge_day2._merge=='left_only') & (rm_df_merge_day2.recog_choice_bool==0)

            rm_df_merge_day2['hit_rate'] = hrs
            rm_df_merge_day2['false_alarm_rate'] = fars
            rm_df_merge_day2['dprime'] = np.nan

            if dp != float("-inf"):
                rm_df_merge_day2['dprime'] = dp    

            # Merge in the image DPRIME 
            rm_df_merge_day2 = rm_df_merge_day2.merge(all_mem_df, on='img_path', how='left', indicator=False)
            rm_df_merge_day2.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 
            rm_df_merge_day2['condition'] = 'day_2'

            full_mem_df = pd.concat([rm_df_merge_day1, rm_df_merge_day2])
            
        else: 
            full_mem_df = rm_df_merge_day1
        
        mb_df.participant=subj
        full_mem_df.participant=subj
        
        learn_df.append(mb_df)
        full_data_df.append(full_mem_df)
        
        
#     ##### Second, process the MEM2 data: 
    elif cell.LongVersion==0:
        relevant_cols = ['img_path', 'prob', 'block', 'Gender',
               'reward', 'participant', 'MEM2_recall_key.keys', 'MEM2_recall_key.rt',
               'MEM2_conf_slider.response', 'MEM2_conf_slider.rt','MEM2_trials.thisTrialN']
        rm_df = df.dropna(subset=['MEM2_trials.thisTrialN'])[relevant_cols].reset_index(drop=True)
        rm_df.rename(columns={'MEM2_recall_key.keys': 'recog_choice',
                     'MEM2_recall_key.rt': 'recog_rt',
                     'MEM2_conf_slider.response': 'conf_choice',
                     'MEM2_conf_slider.rt': 'conf_rt',
                     'MEM2_trials.thisTrialN': 'trials',
                     'Gender': 'gamble_label'},
                     inplace=True)
        mb_df['length'] = 'short'        
        rm_df['length'] = 'short'
    

        rm_df.gamble_label = rm_df.gamble_label.apply(lambda x: 'male' if x==1.0 else 'female')
        # Code reward as 1 or 0
        rm_df.reward = rm_df.reward/100
        # Code recog_choice 
        rm_df['recog_choice_bool'] = rm_df.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
        rm_df.recog_choice = rm_df.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

        rm_df.trials = rm_df.trials.astype(int) + 1
    #     rm_df['age'] = demographics_df[demographics_df.participant==mb_df.participant[0]].age.unique().tolist()*rm_df['trials'].max()
    #     rm_df['sex'] = demographics_df[demographics_df.participant==mb_df.participant[0]].Sex.unique().tolist()*rm_df['trials'].max()

        rm_df_merge = rm_df.merge(mb_df, on='img_path', how='left', indicator=True, suffixes=(None, '_y'))

        # compute the hit-rates and false-alarm rates 
        hits = rm_df_merge[(rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==1)].shape[0]
        misses = rm_df_merge[(rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==0)].shape[0]

        false_alarms = rm_df_merge[(rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==1)].shape[0]
        correct_rejections = rm_df_merge[(rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==0)].shape[0]

        hm = hits+misses
        fc = false_alarms+correct_rejections

        hrs = hits/hm
        fars = false_alarms/fc

        dp = dprime(hrs, fars, hm, fc)

        rm_df_merge['hits'] = (rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==1)
        rm_df_merge['misses'] = (rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==0)
        rm_df_merge['false_alarms'] = (rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==1)
        rm_df_merge['correct_rejections'] = (rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==0)

        rm_df_merge['hit_rate'] = hrs
        rm_df_merge['false_alarm_rate'] = fars
        rm_df_merge['dprime'] = np.nan

        if dp != float("-inf"):
            rm_df_merge['dprime'] = dp    

        # Merge in the image DPRIME 
        rm_df_merge = rm_df_merge.merge(all_mem_df, on='img_path', how='left', indicator=False)
        rm_df_merge.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 
        mb_df = mb_df.merge(all_mem_df, on='img_path', how='left', indicator=False)
        mb_df.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 

        mb_df.participant=subj
        rm_df_merge.participant=subj

        li_mb.append(mb_df)
        li_mem.append(rm_df_merge)

# dm_df = pd.concat(li_mb, axis=0, ignore_index=True)
# dm_df['male'] = 0
# dm_df['female'] = 0
# dm_df.male = dm_df.apply(lambda x: 1 if x.gamble==0 else 0, axis=1)
# dm_df.female = dm_df.apply(lambda x: 1 if x.gamble==1 else 0, axis=1)

# mem_df = pd.concat(li_mem, axis=0, ignore_index=True)
# mem_df.drop(columns=['prob_y',
#        'block_y', 'gamble_label_y', 'reward_y', 'participant_y', 'trials_y', '_merge'], inplace=True)

# # Save these dfs out so I never have to run this stupid code again: 
# dm_df.to_csv(f'{save_dir}/dm_df.csv', index=False)
# mem_df.to_csv(f'{save_dir}/mem_df.csv', index=False)

In [ ]:
learn_df = pd.concat(learn_df)
learn_df['male'] = 0
learn_df['female'] = 0
learn_df.male = learn_df.apply(lambda x: 1 if x.gamble==0 else 0, axis=1)
learn_df.female = learn_df.apply(lambda x: 1 if x.gamble==1 else 0, axis=1)

In [ ]:
# # Save the learn_df 
# learn_df.to_csv(f'{save_dir}/learn_df.csv', index=False)

In [ ]:
full_data_df = pd.concat(full_data_df)

In [ ]:
full_data_df.condition

I'm stupid and accidentally build repeated presentations into my recall data for Day 2. How does this affect Day 2 memory behavior?

In [ ]:
full_data_df.repeated_presentation = full_data_df.repeated_presentation.fillna(0)
sns.catplot(data=full_data_df[full_data_df['_merge']=='both'], 
            x='condition', 
            hue='repeated_presentation', 
            y='hits', kind='bar')
# np.mean(full_data_df[(full_data_df.condition=='day_1') & (full_data_df['_merge']=='both')].hits)



In [ ]:
# What effect does my task code mistake have on the different categories of image in the day 2 memory task?
np.mean(full_data_df[(full_data_df.repeated_presentation==0) & (full_data_df.condition=='day_2') & (full_data_df['_merge']=='both')].hits)
np.mean(full_data_df[(full_data_df.repeated_presentation==1) & (full_data_df.condition=='day_2') & (full_data_df['_merge']=='both')].hits)




How many subjects had no hits, and thus no dprime calculated?

In [ ]:
bad_patients_hits = list(full_data_df[((full_data_df.condition=='day_1') & np.isnan(full_data_df.dprime))].participant.unique())

print(f'{bad_patients_hits} were bad')


How many subjects performed worse than dprime = 0 (chance) on both days?

In [ ]:
bad_patients_dprime = list(set(full_data_df[((full_data_df.condition=='day_1') & (full_data_df.dprime<=0))].participant.unique()))

print(f'{bad_patients_dprime} were bad')

How many subjects performed worse than chance on the decision making task?

In [ ]:
learn_df.groupby('participant').sum().shape

In [ ]:
bad_patients_dm = list(full_data_df[(full_data_df.good_dm_performance==False)].participant.unique())

print(f'{bad_patients_dm} were bad')

In [ ]:
bad_patients = bad_patients_hits + bad_patients_dprime + bad_patients_dm

In [ ]:
full_data_df_good = full_data_df[~full_data_df.participant.isin(bad_patients)]

In [ ]:
full_data_df_good.participant.unique().shape

In [ ]:
subj_level_df_mean = full_data_df_good.groupby(['participant', 'condition']).mean(numeric_only=True).reset_index()


In [ ]:
subj_level_df_mean

In [ ]:
# f, ax = plt.subplots(1,1, dpi=300, figsize=(4,2))
with PdfPages(f'{save_dir}/Fig_BanditBehavior.pdf') as pdf:

    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,2))
    sns.lineplot(data=learn_df, x='trials', y= 'male', color='orange', ax=ax, ci=68)
    sns.lineplot(data=learn_df, x='trials', y= 'female', color='purple', ax=ax, ci=68)
    plt.vlines([16, 33, 48, 65], 0, 1, color = 'black')
    plt.ylabel('p(draw)')
    plt.title(f'subject performance')
    pdf.savefig()
    
    

In [ ]:
sequential_colors = sns.color_palette("BrBG", 2)

In [ ]:
combined_df = full_data_df.merge(learn_df, on=['img_path', 'participant'], how='left', suffixes=['_mem', '_gamble'])
# plot_df = combined_df

In [ ]:
plot_df

In [ ]:
# save_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/plots'
# with PdfPages(f'{save_dir}/memory_behav.pdf') as pdf:
subj_level_df = plot_df.groupby(['participant', 'condition']).mean().reset_index()
# with PdfPages(f'{save_dir_plots}/Fig_MemoryROC.pdf') as pdf:    
f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3))
subj_level_df['false_alarm_rate_bin'] = pd.cut(subj_level_df.false_alarm_rate, bins=np.arange(0, 1.1, .1), right=True)
sns.pointplot(data = subj_level_df[subj_level_df.condition=='day_1'], x='false_alarm_rate_bin', y='hit_rate', 
ax=ax, ci=68, color=sequential_colors[0])
sns.pointplot(data = subj_level_df[subj_level_df.condition=='day_2'], x='false_alarm_rate_bin', y='hit_rate', 
ax=ax, ci=68, color=sequential_colors[1])
ax.set_xlim([-0.5, 9.5])
ax.plot([0, 1], [0, 1], transform=ax.transAxes, color='k')
ax.set_ylim([0, 1])
ax.set(xticks=np.linspace(-0.5, 9.5, 6))
ax.set(xticklabels=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set(xlabel='false alarm rate', ylabel='hit rate')
#     pdf.savefig()


In [ ]:
# with PdfPages(f'{save_dir}/memory_behav_conditions.pdf') as pdf:
#     f, ax = plt.subplots(1,1, dpi=300, figsize=(4,3))
#     sns.boxplot(data=subj_level_df_mean, x='dprime', y='condition', orient='h', palette=sequential_colors,
#                 showcaps=False,boxprops={'facecolor':'None'},
#                     showfliers=False,whiskerprops={'linewidth':0}, ax=ax)
#     sns.stripplot(data=subj_level_df_mean, x='dprime', y='condition', orient='h', palette=sequential_colors, 
#                   size=4, alpha=0.7, jitter=True, ax=ax)

#     plt.vlines(0, plt.ylim()[0],  plt.ylim()[1], color='red')

#     f.tight_layout()
#     pdf.savefig()
    
with PdfPages(f'{save_dir}/emu_dprime_between_days.pdf') as pdf:
    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3))
    sns.pointplot(data=subj_level_df_mean, x='condition', y='dprime', hue='participant', color ='k', ax=ax)
    plt.legend([],[], frameon=False)
    plt.setp(ax.collections, alpha=0) #for the markers
    plt.setp(ax.lines, alpha=.1)       #for the lines
    ax.plot(np.zeros(len(subj_level_df_mean[subj_level_df_mean.condition=='day_1'].dprime)), 
           subj_level_df_mean[subj_level_df_mean.condition=='day_1'].dprime, 'o', color=sequential_colors[0])

    ax.plot(np.ones(len(subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dprime)), 
           subj_level_df_mean[subj_level_df_mean.condition=='day_2'].dprime, 'o', color=sequential_colors[1])

    sns.despine()
    plt.hlines(0, plt.xlim()[0],  plt.xlim()[1], color='red')
    ax.set(xticklabels=['Day 1', 'Day 2'], ylabel="memory (d')")
    pdf.savefig()


## TODO: Hierarchically model the EMU data

In [ ]:
from HierarchicalRW import HierarchicalRWModel
from HierarchicalRW_Fictive1 import HierarchicalRWModel_Fictive1
from HierarchicalRW_Unsigned import HierarchicalRWModel_Unsigned
from HierarchicalBayes import HierarchicalBayesModel
from HierarchicalVKF import HierarchicalVKFModel
from HierarchicalRW_Signed import HierarchicalRWModel_Signed
from HierarchicalWSLS import HierarchicalWSLSModel

In [ ]:
n_subj = learn_df.participant.unique().shape[0]

In [ ]:
actions = learn_df.gamble.values.reshape(n_subj, 80).T
rewards = learn_df.reward.values.reshape(n_subj, 80).T

model = HierarchicalRWModel(n_subj=n_subj)
    
m = model.make_model(actions, rewards)

In [ ]:
pm.model_to_graphviz(m)

In [ ]:
idata = model.fit(m, tune=4000, target_accept=0.9)


In [ ]:
az.to_netcdf(idata, f'{base_dir}/work/qasims01/MemoryBanditData/EMU/HierarchicalRWModel')


In [ ]:
az.summary(idata)

In [ ]:
# # Now, model the data with the best model and save params
# model = RWModel_Fictive1()
# params = {f'{x}': np.nan for x in model.model_params}

# for x in params.keys():
#     full_data_df[f'{x}'] = params[f'{x}']
#     learn_df[f'{x}'] = params[f'{x}']

# full_data_df['rpe'] = np.nan
# learn_df['rpe'] = np.nan

# # Loop through subjects: 
# num_sub = len(full_data_df.participant.unique())
# for ix in range(num_sub):
#     pid = full_data_df.participant.unique()[ix] 
    
#     print(f'Working on subject {pid}')
    
#     actions = learn_df[learn_df.participant==pid].gamble.values
#     rewards = learn_df[learn_df.participant==pid].reward.values   
    
#     # Make the model 
#     m = model.make_model(actions, rewards)
    
# #     Fit the data 
#     idata, callback = model.fit(m, tune=1000, target_accept=0.88, cores=None)
#     if callback.count > 0:
#         idata, callback = model.fit(m, tune=1000, target_accept=0.98, cores=None)

#     model_dict = model.compute_posterior_mean_param(idata)
# #     model_dict = model.compute_posterior_mean_param(idata)
#     for x in params.keys():
#         params[f'{x}'] = model_dict[f'{x}'][0]

#     rpe = model.simulate(n_trials=80, actions=actions, rewards=rewards, params=params) 
    
#     for x in params.keys():
#         full_data_df[f'{x}'][full_data_df.participant==pid] = params[f'{x}']
#         learn_df[f'{x}'][learn_df.participant==pid] = params[f'{x}']
#     # full_data_df['rpe'][(full_data_df.participant==pid) & (full_data_df._merge=='both')] = rpe      
#     learn_df['rpe'][learn_df.participant==pid] = rpe


In [ ]:
# rw_model = RWModel()
# wsls_model = WSLSModel()
# rwfictive1_model = RWModel_Fictive1()
# rwsigned_model = RWModel_Signed()
# bayes_model = BayesModel()
# ewa_model = EWA()
# vkfmodel = VKFModel()

# model_types = {'Bayes': bayes_model,
#                'RWModel': rw_model,
#                'WSLS': wsls_model,
#                'VKF': vkfmodel, 
#                'RWF1': rwfictive1_model,
#                'RWS': rwsigned_model,
#                'EWA': ewa_model}

# # load_dir = f'{base_dir}/projects/guLab/Salman/Prolific/ImmediateMemoryTest'
# # dm_df_all = pd.read_csv(f'{load_dir}/full_data.csv')
# num_sub = len(full_data_df.participant.unique())

# for ix in range(num_sub): 
#     update_progress(ix/num_sub)
#     model_res = {f'{x}':[] for x in model_types.keys()}
#     model_divergences = {f'{x}':[] for x in model_types.keys()}
#     model_params = {f'{x}':[] for x in model_types.keys()}

#     pid = full_data_df.participant.unique()[ix]
#     actions = learn_df[learn_df.participant==pid].gamble.values
#     rewards = learn_df[learn_df.participant==pid].reward.values   
    
#     # Make the model 

#     for model_key, model in model_types.items(): 
#         m = model.make_model(actions, rewards)
#         #     Fit the data 
#         idata, callback = model.fit(m, tune=1000, target_accept=0.88, cores=None)
#         if callback.count > 0:
#             idata, callback = model.fit(m, tune=1000, target_accept=0.98, cores=None)
        
#         model_res[model_key] = idata
#         model_dict = model.compute_posterior_mean_param(idata)
#         model_params[model_key] = model_dict
#         model_divergences[model_key] = callback.count

#     df_comp_loo = az.compare(model_res, ic='waic')
# #     add subject name 
#     df_comp_loo['pid'] = pid
#     df_comp_loo['divergences'] = np.nan
#     for key in model_divergences.keys():
#         df_comp_loo['divergences'][df_comp_loo.index==key] = model_divergences[key]
# #     df_comp_loo['elpd_diff'][df_comp_loo.divergences>0] = df_comp_loo['elpd_diff'].max() + 1
#     df_comp_loo.to_csv(f'{save_dir}/{pid}_model_comp.csv')
#     pd.DataFrame.from_dict(model_params).transpose().to_csv(f'{save_dir}/{pid}_model_fit_params.csv')
    
    


In [ ]:
# save_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/plots'
with PdfPages(f'{save_dir}/EMU_model_comparison.pdf') as pdf:
    
    # aggregate the data
    dfs = []
    for pid in full_data_df.participant.unique():
        df_comp_loo = pd.read_csv(f'{save_dir}/{pid}_model_comp.csv')
        # re-rank the diverging models as tied for worst 
#         df_comp_loo['rank'][df_comp_loo.divergences>0] = 4
#         # re-rank non-diverging models ranked 4th as 3rd 
#         df_comp_loo['rank'][(df_comp_loo.divergences==0) & (df_comp_loo['rank']==4)] = 3
        dfs.append(df_comp_loo)
    df_comp_loos = pd.concat(dfs)
    df_comp_loos.rename(columns={'Unnamed: 0':'model'}, inplace=True)
    df_comp_loos['elpd_waic_raw']= -2*df_comp_loos['elpd_waic'] # comptue the actual information crtierion


    f, ax = plt.subplots(1,1, dpi=300, figsize=(8,4))
    # sns.barplot(data=df_comp_loos[df_comp_loos.divergences==0], x='elpd_waic', y='model', orient='h', errorbar='se',
    #            order = ['Bayes', 'RWF1',  'EWA', 'RWS','RWModel', 'VKF', 'WSLS'], estimator='sum')
    sns.boxplot(data=df_comp_loos, y='elpd_waic_raw', x='model',  order = ['RWF1', 'Bayes',  'VKF', 'WSLS', 'RWS', 'RWModel', 'EWA'], showcaps=False,boxprops={'facecolor':'None'}, showfliers=False,whiskerprops={'linewidth':0})
    sns.stripplot(data=df_comp_loos, y='elpd_waic_raw', x='model', order = ['RWF1', 'Bayes',  'VKF', 'WSLS',  'RWS', 'RWModel', 'EWA'], color='k', size=2, alpha=0.5)
    # Get the handles and labels. For this example it'll be 2 tuples
    # of length 4 each.
    handles, labels = ax.get_legend_handles_labels()
    plt.xlabel('model')
    plt.ylabel("WAIC")
    #     pdf.savefig()

In [ ]:
# Model Frequency: 
# aggregate the data
dfs = []
for pid in full_data_df.participant.unique():
    df_comp_loo = pd.read_csv(f'{save_dir}/{pid}_model_comp.csv')
    # re-rank the diverging models as tied for worst 
#     df_comp_loo['rank'][df_comp_loo.divergences>0] = np.nan
#         # re-rank non-diverging models ranked 4th as 3rd 
#         df_comp_loo['rank'][(df_comp_loo.divergences==0) & (df_comp_loo['rank']==4)] = 3
    dfs.append(df_comp_loo)
df_comp_loos = pd.concat(dfs)
df_comp_loos.rename(columns={'Unnamed: 0':'model'}, inplace=True)

# Process the divergent models by setting them to the worst rank and moving everything else down a rank
# df_comp_loo['rank'][df_comp_loo.divergences>0] = np.nan
df_comp_loos['rank'][df_comp_loos.model.isin(['RWModel', 'RWS', 'EWA', 'VKF'])] = np.nan

grouped_df = df_comp_loos.sort_values('rank').groupby('pid').apply(pd.DataFrame.head, n=1).drop(columns=['pid']).reset_index()


In [ ]:

f, ax = plt.subplots(1,1, dpi=300, figsize=(5,3))
sns.countplot(data=grouped_df, x='model')



In [ ]:
ax = sns.jointplot(x=np.mean(np.mean(rw_idata.posterior['alpha'].values, axis=0), axis=0), 
                   y=np.mean(np.mean(rw_idata.posterior['beta'].values, axis=0), axis=0), 
                   height=8, ratio=2, kind='kde', shade=True, cmap='Spectral', 
                   fill=True, cbar=True)

plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# get the current positions of the joint ax and the ax for the marginal x
pos_joint_ax = ax.ax_joint.get_position()
pos_marg_x_ax = ax.ax_marg_x.get_position()
# reposition the joint ax so it has the same width as the marginal x ax
ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# reposition the colorbar using new x positions and y positions of the joint ax
ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)

In [ ]:
# Now, model the data with the best model and save params
model = RWModel_Fictive1()

full_data_df['alpha'] = np.nan
full_data_df['beta'] = np.nan
# full_data_df['rpe']= np.nan

learn_df['alpha'] = np.nan
learn_df['beta'] = np.nan
learn_df['rpe']= np.nan

# Loop through subjects: 
num_sub = len(full_data_df.participant.unique())
for ix in range(num_sub):
    pid = full_data_df.participant.unique()[ix] 
    
    actions = learn_df[learn_df.participant==pid].gamble.values
    rewards = learn_df[learn_df.participant==pid].reward.values   
    
    # Make the model 
    m = model.make_model(actions, rewards)
    
#     Fit the data 
    idata, callback = model.fit(m, tune=1500, target_accept=0.95, cores=None)
    model_dict = model.compute_posterior_mean_param(idata)
#     model_dict = model.compute_posterior_mean_param(idata)
    params = {'alpha': model_dict['alpha'][0],
             'beta': model_dict['beta'][0]}

    rpe = model.simulate(n_trials=80, actions=actions, rewards=rewards, params=params) 

    full_data_df['alpha'][full_data_df.participant==pid] = params['alpha']
    full_data_df['beta'][full_data_df.participant==pid] = params['beta']
#     full_data_df['rpe'][full_data_df.participant==pid] = rpe
    
    learn_df['alpha'][learn_df.participant==pid] = params['alpha']
    learn_df['beta'][learn_df.participant==pid] = params['beta']
#     learn_df['prpe'][learn_df.participant==pid] = prpe
#     learn_df['nrpe'][learn_df.participant==pid] = nrpe
    learn_df['rpe'][learn_df.participant==pid] = rpe
 
full_data_df = full_data_df.merge(learn_df[['participant', 'img_path', 'rpe']], on=['participant', 'img_path'], how='left')
full_data_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/full_data_emu_RWF1.csv')
learn_df.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/learn_df_emu_RWF1.csv')


In [ ]:
combined_df = full_data_df.merge(learn_df, on=['img_path', 'participant'], how='left', suffixes=['_mem', '_gamble'])


In [ ]:
combined_df['rpe']= combined_df['rpe_gamble']

In [ ]:
combined_df.drop(columns=['rpe_mem', 'rpe_gamble'], inplace=True)

In [ ]:
list(combined_df.keys())

In [ ]:
learn_df

In [ ]:
#UI007 is missing a feedback_start time for some godforsaken reason

In [ ]:
learn_df.loc[learn_df.feedback_start=='None', 'feedback_start'] = learn_df.loc[learn_df.feedback_start=='None', 'baseline_start']-1.5

combined_df.loc[combined_df.feedback_start_gamble=='None', 'feedback_start_gamble'] = combined_df.loc[combined_df.feedback_start_gamble=='None', 'baseline_start_gamble']-1.5




In [ ]:
full_data_df.to_csv(f'{save_dir}/full_df.csv', index=False)
learn_df.to_csv(f'{save_dir}/learn_df.csv', index=False)

In [ ]:
full_data_df = pd.read_csv(f'{save_dir}/full_df.csv')
learn_df = pd.read_csv(f'{save_dir}/learn_df.csv')

In [ ]:
combined_df.to_csv(f'{save_dir}/combined_df.csv', index=False)

In [ ]:
combined_df.condition

In [ ]:
ax = sns.jointplot(data = learn_df.groupby('participant').mean(), 
                   x='alpha', y='beta', height=8, ratio=2, kind='kde', shade=True, cmap='Spectral', 
                   fill=True, cbar=True)

plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# get the current positions of the joint ax and the ax for the marginal x
pos_joint_ax = ax.ax_joint.get_position()
pos_marg_x_ax = ax.ax_marg_x.get_position()
# reposition the joint ax so it has the same width as the marginal x ax
ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# reposition the colorbar using new x positions and y positions of the joint ax
ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)

In [ ]:
combined_df = full_data_df.merge(learn_df, on=['img_path', 'participant'], how='left', suffixes=['_mem', '_gamble'])
for ix, row in learn_df.iterrows():
    combined_df.loc[(combined_df.img_path==row.img_path) & (combined_df.participant==row.participant), 'rpe'] = row.rpe
    
plot_df = combined_df

In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(2,2))
sns.regplot(data=plot_df[plot_df.condition=='day_1'], x='rpe', y='hits', logistic=True, ci=68, n_boot=100, color=sequential_colors[0])
sns.regplot(data=plot_df[plot_df.condition=='day_2'], x='rpe', y='hits', logistic=True, ci=68, n_boot=100, color=sequential_colors[1])

ax.set_ylim([0.4, .6])
ax.set_ylabel('p(hit)')

In [ ]:
with PdfPages(f'{save_dir}/Fig_MemoryRPECorr.pdf') as pdf:    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(2,2))
    sns.regplot(data=combined_df, x='rpe', y='hits', color = 'black',  x_bins = np.linspace(-1,1,8), logistic=True, ci=95, n_boot=500, scatter_kws={'s':20})
#     ax.set_ylim([0.45, .75])
    ax.set_ylabel('p(hit)')
#     pdf.savefig()
    
    

In [ ]:
# f, ax = plt.subplots(1,1, dpi=300, figsize=(2,2))
sns.lmplot(data=combined_df, x='rpe', y='hits', hue = 'participant', logistic=True, ci=None)
# sns.regplot(data=combined_df[combined_df.participant=='MS015'], x='rpe', y='hits', color='red', logistic=True, ci=None)
# ax.set_ylim([0.2, 0.7])
ax.set_ylabel('p(hit)')


In [ ]:
#  Model different models, do the WAIC, and save the d_loo for each model and subject
RW = RWModel()
RWB = RWModelBias()
RWF1 = RWModel_Fictive1()
RWF2 = RWModel_Fictive2()
RWS = RWModel_Signed()
RWPM = RWModelPMA()
RWEWA = EWA()

# dm_df_all['alpha'] = np.nan
# dm_df_all['beta'] = np.nan
# dm_df_all['rpe']= np.nan

model_types = {'RW':RW,
         'RWB':RWB,
         'RWF1':RWF1,
         'RWF2':RWF2,
         'RWS':RWS,
         'RWPM':RWPM,
         'RWEWA':RWEWA}

model_res = {f'{x}':[] for x in model_types.keys()}

d_loos = []

# Loop through subjects: 
num_sub = len(dm_df[dm_df['good_dm_performance']].participant.unique())
for ix in range(num_sub):
    pid = dm_df[dm_df['good_dm_performance']].participant.unique()[ix]
    
    actions = dm_df[dm_df.participant==pid].gamble.values
    rewards = dm_df[dm_df.participant==pid].reward.values        
    # Loop through models:
    for model_key, model in model_types.items(): 
        if model_key =='RWPM':
            pmi = dm_df[dm_df.participant==pid].image_dprime.values
            m = model.make_model(actions, rewards, pmi)
        else:
            m = model.make_model(actions, rewards)
        try:
            idata = model.fit(model=m, tune=1000, target_accept=0.9)
        except: 
            model_res[model_key] = np.nan
            continue
        model_res[model_key] = pm.waic(idata, model).waic
#         idata
#         waics.append(pm.waic(idata, model).waic)

        
#     df_comp_loo = az.compare(model_res, ic='waic')
    df_loo = pd.DataFrame(columns=['pid', 'model', 'd_waic'])
    df_loo['model'] = list(model_res.keys())
    df_loo['pid'] = pid
    d_waics = []
    for key, val in model_res.items():
        d_waics.append(val - max(model_res.values()))
    df_loo['d_waic'] = d_waics
    d_loos.append(df_loo)

all_model_comps = pd.concat(d_loos)
all_model_comps.to_csv(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/all_model_comps.csv', index=False)
#     model_dict = model.compute_posterior_mean_param(idata)
    
#     # fill parameters
#     for param_key in model_dict.keys():
#         dm_df_all[f'{param_key}'][dm_df_all.participant==pid] = model_dict[f'{param_key}'][0]
    
#     # compute RPE
#     rpe = model.compute_latent_variables(dm_df_all[dm_df_all.participant==pid].gamble.values, 
#                                          dm_df_all[dm_df_all.participant==pid].reward.values,
#                                          model_dict)
    
#     dm_df_all['rpe'][dm_df_all.participant==pid] = rpe

In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(8,4))
sns.boxplot(data=all_model_comps, x='d_waic', y='model', orient='h', order=['RWF1', 'RW', 'RWEWA', 'RWS', 'RWPM', 'RWB', 'RWF2'],
            showcaps=False,boxprops={'facecolor':'None'},
                showfliers=False,whiskerprops={'linewidth':0})
sns.stripplot(data=all_model_comps, x='d_waic', y='model', orient='h', order=['RWF1', 'RW', 'RWEWA', 'RWS', 'RWPM', 'RWB', 'RWF2'],
              color='k', size=2, alpha=0.75)
# Get the handles and labels. For this example it'll be 2 tuples
# of length 4 each.
handles, labels = ax.get_legend_handles_labels()
plt.ylabel('model')
plt.xlabel(u'\u0394 waic')

# Now let's look at little bit at the day two performance: 

In [ ]:
subj_list_long = ['UI001', 'UI002'] # 'MS015', 'MS016', 'MS017']
li_mem24 = []
for subj in subj_list_long:
    
    mb_df = dm_df[dm_df.participant==subj]
    
    two_ab_ntrials = 80
    all_mem_file =f'{load_dir}/all_mem_data_long.xlsx' 
    all_mem_df = pd.read_excel(all_mem_file, engine='openpyxl')
    all_mem_df = all_mem_df[['img_path', 'DPRIME']]
    all_mem_df.img_path = all_mem_df.img_path.apply(lambda x: x[:-4]+'.png' )

    subj_beh_path = f'{load_dir}/{subj}/behav/Day2'
    # Look at actual data (for one subject -- later expand to multiple subjects)
    subj_beh_file = glob.glob(f'{subj_beh_path}/*.csv')[0]
    
    rm_df = pd.read_csv(subj_beh_file, index_col=None, header=0)
    rm_df.rename(columns={'trialIm': 'img_path'}, inplace=True)  
        
    relevant_cols = ['img_path', 'participant', 'key_resp.keys', 'key_resp.rt',
               'new_slider.rating', 'new_slider.rt','trials.thisN']
    rm_df = rm_df.dropna(subset=['trials.thisN'])[relevant_cols].reset_index(drop=True)

    rm_df.rename(columns={'key_resp.keys': 'recog_choice',
                 'key_resp.rt': 'recog_rt',
                 'new_slider.rating': 'conf_choice',
                 'new_slider.rt': 'conf_rt',
                 'trials.thisN': 'trials'},
                 inplace=True) 
    
    # Code recog_choice 
    rm_df['recog_choice_bool'] = rm_df.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
    rm_df.recog_choice = rm_df.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

    rm_df.trials = rm_df.trials.astype(int) + 1
#     rm_df['age'] = demographics_df[demographics_df.participant==mb_df.participant[0]].age.unique().tolist()*rm_df['trials'].max()
#     rm_df['sex'] = demographics_df[demographics_df.participant==mb_df.participant[0]].Sex.unique().tolist()*rm_df['trials'].max()

    rm_df_merge = rm_df.merge(mb_df, on='img_path', how='left', indicator=True, suffixes=(None, '_y'))

    # compute the hit-rates and false-alarm rates 
    hits = rm_df_merge[(rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==1)].shape[0]
    misses = rm_df_merge[(rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==0)].shape[0]

    false_alarms = rm_df_merge[(rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==1)].shape[0]
    correct_rejections = rm_df_merge[(rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==0)].shape[0]

    hm = hits+misses
    fc = false_alarms+correct_rejections

    hrs = hits/hm
    fars = false_alarms/fc

    dp = dprime(hrs, fars, hm, fc)

    rm_df_merge['hits'] = (rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==1)
    rm_df_merge['misses'] = (rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==0)
    rm_df_merge['false_alarms'] = (rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==1)
    rm_df_merge['correct_rejections'] = (rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==0)
    
    rm_df_merge['hit_rate'] = hrs
    rm_df_merge['false_alarm_rate'] = fars
    rm_df_merge['dprime'] = np.nan
    
    if dp != float("-inf"):
        rm_df_merge['dprime'] = dp    

    # Merge in the image DPRIME 
    rm_df_merge = rm_df_merge.merge(all_mem_df, on='img_path', how='left', indicator=False)
    rm_df_merge.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 

    rm_df_merge.participant=subj

    li_mem24.append(rm_df_merge)

mem_df24 = pd.concat(li_mem24, axis=0, ignore_index=True)
mem_df24.drop(columns=['participant_y', 'trials_y', '_merge'], inplace=True)

# Save these dfs out so I never have to run this stupid code again: 
mem_df24.to_csv(f'{save_dir}/mem_df24.csv', index=False)



In [ ]:
mem_df24

In [ ]:
# f, ax = plt.subplots(1,1, dpi=300, figsize=(2,2))
sns.lmplot(data=mem_df24, x='rpe', y='hits', hue = 'participant', logistic=True, ci=None)
# sns.regplot(data=combined_df[combined_df.participant=='MS015'], x='rpe', y='hits', color='red', logistic=True, ci=None)
# ax.set_ylim([0.2, 0.7])
ax.set_ylabel('p(hit)')


In [ ]:
mem_df24

In [ ]:
# merge the two dfs and make a point plot comparing Day1 and Day2 memory 